In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Metric summary and top-20 kpi_name counts
metric_summary_df = pd.DataFrame([{
    "file": "metric.csv",
    "total_rows": len(df_metric),
    "unique_cmdb_ids": sorted(df_metric["cmdb_id"].dropna().unique().tolist()),
    "min_timestamp_utc": df_metric["timestamp_dt"].min(),
    "max_timestamp_utc": df_metric["timestamp_dt"].max()
}])
metric_top_kpis = (
    df_metric["kpi_name"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "kpi_name", "kpi_name": "count"})
    .head(20)
)

# Trace summary and top-20 trace_name counts
trace_summary_df = pd.DataFrame([{
    "file": "trace.csv",
    "total_rows": len(df_trace),
    "unique_cmdb_ids": sorted(df_trace["cmdb_id"].dropna().unique().tolist()),
    "min_timestamp_utc": df_trace["timestamp_dt"].min(),
    "max_timestamp_utc": df_trace["timestamp_dt"].max()
}])
trace_top_traces = (
    df_trace["trace_name"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "trace_name", "trace_name": "count"})
    .head(20)
)

# Log summary and top-20 log_name counts
log_summary_df = pd.DataFrame([{
    "file": "log.csv",
    "total_rows": len(df_log),
    "unique_cmdb_ids": sorted(df_log["cmdb_id"].dropna().unique().tolist()),
    "min_timestamp_utc": df_log["timestamp_dt"].min(),
    "max_timestamp_utc": df_log["timestamp_dt"].max()
}])
log_top_logs = (
    df_log["log_name"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "log_name", "log_name": "count"})
    .head(20)
)

# Error logs summary (no top list requested)
error_summary_df = pd.DataFrame([{
    "file": "error_logs.csv",
    "total_rows": len(df_error),
    "unique_cmdb_ids": sorted(df_error["cmdb_id"].dropna().unique().tolist()),
    "min_timestamp_utc": df_error["timestamp_dt"].min(),
    "max_timestamp_utc": df_error["timestamp_dt"].max()
}])

# Display compact summaries and top-20 lists
metric_summary_df, metric_top_kpis, trace_summary_df, trace_top_traces, log_summary_df, log_top_logs, error_summary_df

```
Out[1]:
```
Summary of the loaded telemetry files:

- metric.csv
  - Total rows: 1850
  - Unique cmdb_id values (sampleed): [carts, carts-db, catalogue, catalogue-db, ...]
  - Time range (UTC): 2024-01-20 09:57:00+00:00 — 2024-01-20 10:21:00+00:00
  - Top kpi_name (top 20 shown; here top entries):
    - cpu: 375
    - socket: 375
    - mem: 375
    - diskio: 200
    - latency-50: 175
    - latency-90: 175
    - workload: 175

- trace.csv
  - Total rows: 0
  - Unique cmdb_id values: []
  - Time range (UTC): none (no timestamps)
  - Top trace_name: none (no data)

- log.csv
  - Total rows: 454
  - Unique cmdb_id values (sampled): [carts, carts-db, catalogue, front-end, orders, ...]
  - Time range (UTC): 2024-01-20 09:57:00+00:00 — 2024-01-20 10:21:00+00:00
  - Top log_name (top 20 shown; here top entries):
    - log.error_count: 227
    - log.row_count: 227

- error_logs.csv
  - Total rows: 0
  - Unique cmdb_id values: []
  - Time range (UTC): none (no timestamps)

Notes: trace.csv and error_logs.csv contain no rows. Metric and log data cover the same UTC interval (2024-01-20 09:57 to 10:21).

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows                                    unique_cmdb_ids         min_timestamp_utc         max_timestamp_utc
0  metric.csv        1850  [carts, carts-db, catalogue, catalogue-db, fro... 2024-01-20 09:57:00+00:00 2024-01-20 10:21:00+00:00,         count  count
0         cpu    375
1      socket    375
2         mem    375
3      diskio    200
4  latency-50    175
5  latency-90    175
6    workload    175,         file  total_rows unique_cmdb_ids min_timestamp_utc max_timestamp_utc
0  trace.csv           0              []               NaT               NaT, Empty DataFrame
Columns: [count, count]
Index: [],       file  total_rows                                    unique_cmdb_ids         min_timestamp_utc         max_timestamp_utc
0  log.csv         454  [carts, carts-db, catalogue, front-end, orders... 2024-01-20 09:57:00+00:00 2024-01-20 10:21:00+00:00,              count  count
0  log.error_count    227
1    log.row_count    227,              file  total_rows unique_cmdb_ids min_timestamp_utc max_timestamp_utc
0  error_logs.csv           0              []               NaT               NaT)```
```

In [ ]:
# In[2]:

# Reuse df_metric loaded previously
import numpy as np
components = ["carts","catalogue","orders","payment","user"]

# Ensure timestamp_dt exists and is UTC-aware
if "timestamp_dt" not in df_metric.columns:
    df_metric["timestamp_dt"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Filter to the requested service components (but do NOT pre-filter for percentile calculations)
df_comp = df_metric[df_metric["cmdb_id"].isin(components)].copy()

# 1) Compute count_total, P95, P05 per (cmdb_id, kpi_name) using full group's rows
grp_stats = (
    df_comp
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(count_total="size",
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Define incident window (UTC) and filter rows within window (inclusive)
start = pd.to_datetime("2024-01-20 09:54:48", utc=True)
end = pd.to_datetime("2024-01-20 10:24:48", utc=True)
df_window = df_comp[(df_comp["timestamp_dt"] >= start) & (df_comp["timestamp_dt"] <= end)].copy()

# Merge group-level stats into windowed rows to detect anomalies using global thresholds
df_w = df_window.merge(grp_stats, on=["cmdb_id", "kpi_name"], how="left")

# Mark high/low anomalies within the window using pre-computed p95/p05
df_w["high_bool"] = df_w["value"] >= df_w["p95"]
df_w["low_bool"] = df_w["value"] <= df_w["p05"]

# Aggregate window statistics per group
count_in_window = df_w.groupby(["cmdb_id", "kpi_name"])["value"].size().reset_index(name="count_in_window")
high_counts = df_w[df_w["high_bool"]].groupby(["cmdb_id", "kpi_name"])["high_bool"].sum().reset_index(name="high_anomaly_count")
low_counts = df_w[df_w["low_bool"]].groupby(["cmdb_id", "kpi_name"])["low_bool"].sum().reset_index(name="low_anomaly_count")
earliest_high = df_w[df_w["high_bool"]].groupby(["cmdb_id", "kpi_name"])["timestamp_dt"].min().reset_index(name="earliest_high_ts_utc")
earliest_low = df_w[df_w["low_bool"]].groupby(["cmdb_id", "kpi_name"])["timestamp_dt"].min().reset_index(name="earliest_low_ts_utc")

# Combine all stats into a single result table, keeping groups even if they had no rows in the window
res = grp_stats.merge(count_in_window, on=["cmdb_id", "kpi_name"], how="left")
res = res.merge(high_counts, on=["cmdb_id", "kpi_name"], how="left")
res = res.merge(low_counts, on=["cmdb_id", "kpi_name"], how="left")
res = res.merge(earliest_high, on=["cmdb_id", "kpi_name"], how="left")
res = res.merge(earliest_low, on=["cmdb_id", "kpi_name"], how="left")

# Fill missing counts with 0 and ensure integer types for count columns
res["count_in_window"] = res["count_in_window"].fillna(0).astype(int)
res["high_anomaly_count"] = res["high_anomaly_count"].fillna(0).astype(int)
res["low_anomaly_count"] = res["low_anomaly_count"].fillna(0).astype(int)

# Sort by the larger of high/low anomaly counts descending and limit to top 20 rows
res["max_anom"] = res[["high_anomaly_count", "low_anomaly_count"]].max(axis=1)
result_df = (
    res
    .sort_values(by="max_anom", ascending=False)
    .drop(columns=["max_anom"])
    .loc[:, ["cmdb_id", "kpi_name", "count_total", "p95", "p05",
             "count_in_window", "high_anomaly_count", "low_anomaly_count",
             "earliest_high_ts_utc", "earliest_low_ts_utc"]]
    .head(20)
)

result_df

```
Out[2]:
```
summary = (
    "Analysis of metric.csv for components [carts, catalogue, orders, payment, user] "
    "within the incident window 2024-01-20 09:54:48 — 2024-01-20 10:24:48 UTC:\n\n"
    "General: Each reported (cmdb_id, kpi_name) group has 25 total samples and 25 samples in the window.\n\n"
    "Top affected groups (by max(high_anomaly_count, low_anomaly_count)):\n"
    "1) catalogue / socket: 25 high and 25 low anomalies (all values equal to the thresholds). "
    "Earliest high/low: 2024-01-20 09:57:00+00:00.\n"
    "2) orders / diskio: 2 high, 23 low. Earliest high: 2024-01-20 09:57:00+00:00. Earliest low: 2024-01-20 09:59:00+00:00.\n"
    "3) payment / socket: 2 high, 23 low. Earliest high: 2024-01-20 10:20:00+00:00. Earliest low: 2024-01-20 09:57:00+00:00.\n"
    "4) user / socket: 6 high, 2 low. Earliest high: 2024-01-20 10:16:00+00:00. Earliest low: 2024-01-20 09:57:00+00:00.\n"
    "5) orders / socket and carts / socket: each shows more high than low anomalies (orders/socket 2 high/3 low; carts/socket 3 high/2 low) with earliest highs around 10:01–10:13 UTC.\n\n"
    "Interpretation: catalogue/socket stands out with every point flagged as both high and low relative to its global P95/P05 (indicating constant values equal to those thresholds). "
    "orders.diskio and payment.socket show many low anomalies (values at or below P05). user.socket shows multiple high spikes. These groups are the primary candidates for further investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_total           p95           p05  count_in_window  high_anomaly_count  low_anomaly_count      earliest_high_ts_utc       earliest_low_ts_utc
10  catalogue      socket           25  6.000000e+00  6.000000e+00               25                  25                 25 2024-01-20 09:57:00+00:00 2024-01-20 09:57:00+00:00
13     orders      diskio           25  2.601830e+04  0.000000e+00               25                   2                 23 2024-01-20 09:57:00+00:00 2024-01-20 09:59:00+00:00
23    payment      socket           25  4.066667e+00  4.000000e+00               25                   2                 23 2024-01-20 10:20:00+00:00 2024-01-20 09:57:00+00:00
29       user      socket           25  1.900000e+01  1.720000e+01               25                   6                  2 2024-01-20 10:16:00+00:00 2024-01-20 09:57:00+00:00
17     orders      socket           25  2.210000e+01  1.431667e+01               25                   2                  3 2024-01-20 10:13:00+00:00 2024-01-20 10:06:00+00:00
4       carts      socket           25  1.200000e+01  1.058667e+01               25                   3                  2 2024-01-20 09:57:00+00:00 2024-01-20 10:05:00+00:00
2       carts  latency-90           25  1.338379e-01  2.367497e-02               25                   2                  2 2024-01-20 10:17:00+00:00 2024-01-20 09:58:00+00:00
6   catalogue         cpu           25  1.801182e-01  1.606993e-01               25                   2                  2 2024-01-20 10:03:00+00:00 2024-01-20 10:14:00+00:00
5       carts    workload           25  8.614170e+00  7.652137e+00               25                   2                  2 2024-01-20 10:03:00+00:00 2024-01-20 09:57:00+00:00
8   catalogue  latency-90           25  4.767537e-03  4.625798e-03               25                   2                  2 2024-01-20 09:58:00+00:00 2024-01-20 10:12:00+00:00
7   catalogue  latency-50           25  3.093076e-03  3.014332e-03               25                   2                  2 2024-01-20 09:58:00+00:00 2024-01-20 10:12:00+00:00
9   catalogue         mem           25  6.189739e+06  6.048495e+06               25                   2                  2 2024-01-20 10:20:00+00:00 2024-01-20 09:57:00+00:00
3       carts         mem           25  2.177485e+08  2.143158e+08               25                   2                  2 2024-01-20 10:10:00+00:00 2024-01-20 10:19:00+00:00
1       carts  latency-50           25  1.929503e-02  1.432997e-02               25                   2                  2 2024-01-20 10:20:00+00:00 2024-01-20 09:58:00+00:00
0       carts         cpu           25  3.505097e+00  1.368618e+00               25                   2                  2 2024-01-20 10:01:00+00:00 2024-01-20 10:19:00+00:00
14     orders  latency-50           25  1.145421e+00  3.616321e-02               25                   2                  2 2024-01-20 10:17:00+00:00 2024-01-20 10:08:00+00:00
12     orders         cpu           25  2.829850e+00  1.101655e+00               25                   2                  2 2024-01-20 10:01:00+00:00 2024-01-20 10:18:00+00:00
11  catalogue    workload           25  4.265857e+00  3.820070e+00               25                   2                  2 2024-01-20 10:00:00+00:00 2024-01-20 10:02:00+00:00
15     orders  latency-90           25  2.229084e+00  4.723338e-02               25                   2                  2 2024-01-20 10:17:00+00:00 2024-01-20 10:08:00+00:00
19    payment         cpu           25  9.678674e-02  8.582246e-02               25                   2                  2 2024-01-20 09:58:00+00:00 2024-01-20 10:13:00+00:00```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing df_metric and df_log; ensure timestamp_dt exists
if "timestamp_dt" not in df_metric.columns:
    df_metric["timestamp_dt"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
if "timestamp_dt" not in df_log.columns:
    df_log["timestamp_dt"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

components = ["carts","catalogue","orders","payment","user"]

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 09:54:48", utc=True)
end = pd.to_datetime("2024-01-20 10:24:48", utc=True)

# 1) Metric anomaly segmentation
# Compute global P95 and P05 per (cmdb_id, kpi_name) using the full series (no window filter)
metric_comp_full = df_metric[df_metric["cmdb_id"].isin(components)].copy()
grp_thresholds = (
    metric_comp_full
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(count_total="size",
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Filter metric rows to the incident window
metric_window = metric_comp_full[(metric_comp_full["timestamp_dt"] >= start) & (metric_comp_full["timestamp_dt"] <= end)].copy()

# Merge thresholds into windowed rows
metric_window = metric_window.merge(grp_thresholds, on=["cmdb_id", "kpi_name"], how="left")

# Mark anomaly type per rule (high if value >= p95, else low if value <= p05, else normal)
metric_window["anomaly_type"] = np.where(
    metric_window["value"] >= metric_window["p95"], "high",
    np.where(metric_window["value"] <= metric_window["p05"], "low", "normal")
)

# Sort for sequence detection
metric_window = metric_window.sort_values(["cmdb_id", "kpi_name", "timestamp_dt"]).reset_index(drop=True)

# Detect segment boundaries within each (cmdb_id,kpi_name)
metric_window["prev_type"] = metric_window.groupby(["cmdb_id", "kpi_name"])["anomaly_type"].shift()
metric_window["is_change"] = (metric_window["anomaly_type"] != metric_window["prev_type"])
# Start of first row of a group should be counted as change True
metric_window["is_change"] = metric_window["is_change"].fillna(True)
# Segment id per group
metric_window["seg_id"] = metric_window.groupby(["cmdb_id", "kpi_name"])["is_change"].cumsum()

# Aggregate segments, but keep only non-normal anomaly segments
seg_groups = metric_window[metric_window["anomaly_type"].isin(["high", "low"])].groupby(["cmdb_id", "kpi_name", "seg_id", "anomaly_type"])

metrics_sequences = (
    seg_groups.agg(
        start_ts_utc=("timestamp_dt", "min"),
        end_ts_utc=("timestamp_dt", "max"),
        length_points=("timestamp_dt", "size"),
        max_value=("value", "max"),
        min_value=("value", "min"),
        p95=("p95", "first"),
        p05=("p05", "first")
    )
    .reset_index(drop=False)
)

# Compute relative_breach per specification, handling division by zero
def compute_rel(row):
    if row["anomaly_type"] == "high":
        mv = row["max_value"]
        p95 = row["p95"]
        return np.nan if mv == 0 else (mv - p95) / mv
    else:  # low
        p05 = row["p05"]
        mn = row["min_value"]
        return np.nan if p05 == 0 else (p05 - mn) / p05

metrics_sequences["relative_breach"] = metrics_sequences.apply(compute_rel, axis=1)

# Final column order and sorting by length_points desc, then top 20
metrics_sequences = metrics_sequences.sort_values(by="length_points", ascending=False)
metrics_sequences = metrics_sequences.loc[:, ["cmdb_id", "kpi_name", "anomaly_type",
                                             "start_ts_utc", "end_ts_utc",
                                             "length_points", "max_value", "min_value",
                                             "p95", "p05", "relative_breach"]].head(20)

# 2) Log error summary
# Filter logs to components and incident window
log_window = df_log[df_log["cmdb_id"].isin(components) & (df_log["timestamp_dt"] >= start) & (df_log["timestamp_dt"] <= end)].copy()

# total rows per cmdb_id
total_rows = log_window.groupby("cmdb_id").size().rename("total_rows_in_window")

# sum_error_count where log_name == "log.error_count"
sum_error = log_window[log_window["log_name"] == "log.error_count"].groupby("cmdb_id")["value"].sum().rename("sum_error_count")

# earliest_error_ts_utc where log_name == "log.error_count" and value > 0
earliest_error = (
    log_window[(log_window["log_name"] == "log.error_count") & (log_window["value"] > 0)]
    .groupby("cmdb_id")["timestamp_dt"].min()
    .rename("earliest_error_ts_utc")
)

# sum_row_count where log_name == "log.row_count"
sum_row_count = log_window[log_window["log_name"] == "log.row_count"].groupby("cmdb_id")["value"].sum().rename("sum_row_count")

# Combine into one DataFrame containing only the requested components (to include zeros if missing)
log_summary = pd.DataFrame({"cmdb_id": components}).set_index("cmdb_id")
log_summary = log_summary.join(total_rows).join(sum_error).join(earliest_error).join(sum_row_count)

# Fill missing numeric sums with 0, keep earliest_error_ts_utc as NaT if missing
log_summary["total_rows_in_window"] = log_summary["total_rows_in_window"].fillna(0).astype(int)
log_summary["sum_error_count"] = log_summary["sum_error_count"].fillna(0)
log_summary["sum_row_count"] = log_summary["sum_row_count"].fillna(0)

log_summary = log_summary.reset_index().loc[:, ["cmdb_id", "total_rows_in_window", "sum_error_count", "earliest_error_ts_utc", "sum_row_count"]]

# Sort by sum_error_count desc and limit to top 20 (components list is small)
log_summary = log_summary.sort_values(by="sum_error_count", ascending=False).head(20)

# Output the two compact result tables: metrics_sequences then log_summary
metrics_sequences, log_summary

```
Out[3]:
```
Metrics anomaly sequences (top findings)
- catalogue / socket: 1 long "high" sequence spanning the whole window (2024-01-20 09:57 — 10:21 UTC), length 25 points. max=min=6.0, p95=p05=6.0. (All values equal to threshold → flagged as both high/low in other views.)
- orders / diskio: large "low" sequence (2024-01-20 09:59 — 10:21 UTC), length 23 points, all zeros (min=max=0). p95=26018.30, p05=0 → relative breach for low is NaN (division by zero). Also a short "high" 2-point spike earlier (09:57—09:58) with large relative breach (~0.8587).
- payment / socket: long "low" sequence (09:57 — 10:19 UTC), length 23 points, values ~4.0, p95≈4.0667, p05=4.0 (relative_breach 0.0).
- user / socket: short "high" run (10:16 — 10:21 UTC), length 6 points, constant value 19.0, p95=19.0, p05=17.2 (relative_breach 0.0).
- carts / cpu: two short sequences (high 10:01—10:02, length 2, relative breach ~0.3521; low 10:19—10:20, length 2, relative breach ~0.0670).
- Many other sequences are short (mostly length 2) across latency, mem, workload, etc., with small relative breaches.

Interpretation from metrics:
- catalogue/socket stands out as abnormal because it is constant at the threshold across the entire window (likely a sensor/metric plateau or threshold artifact).
- orders.diskio shows sustained low values (zeros) for most of the window — likely reduced disk activity or missing/invalid disk metric samples; a brief high spike exists prior to the sustained low.
- payment.socket has prolonged low values around its P05.
- user.socket exhibits a short run of high values (spike) mid/late window.
- carts.cpu shows short high and low episodes.

Log summary (window 2024-01-20 09:54:48 — 10:24:48 UTC)
- carts: 46 log rows, sum_error_count = 0, earliest_error = none, sum_row_count = 1400
- catalogue: 50 rows, sum_error_count = 0, earliest_error = none, sum_row_count = 3874
- orders: 50 rows, sum_error_count = 0, earliest_error = none, sum_row_count = 2908
- payment: 50 rows, sum_error_count = 0, earliest_error = none, sum_row_count = 2414
- user: 50 rows, sum_error_count = 0, earliest_error = none, sum_row_count = 14053

Interpretation from logs:
- No error_count > 0 events recorded for these components in the window. Logs do not show recorded errors to correlate with metric anomalies.

Overall conclusion / suggested next steps
- Prioritize investigating catalogue/socket (constant at threshold) and orders.diskio (sustained zero/low diskio) as primary unusual signals. Also inspect payment.socket (sustained low) and user/socket (short high spike).
- Because log.error_count is zero for these services in the window, follow up with service-level traces, container/process status, metric source integrity (are metrics missing or clipped?), and disk I/O collection correctness for orders.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name anomaly_type              start_ts_utc                end_ts_utc  length_points     max_value     min_value           p95           p05  relative_breach
28  catalogue      socket         high 2024-01-20 09:57:00+00:00 2024-01-20 10:21:00+00:00             25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00         0.000000
37     orders      diskio          low 2024-01-20 09:59:00+00:00 2024-01-20 10:21:00+00:00             23  0.000000e+00  0.000000e+00  2.601830e+04  0.000000e+00              NaN
65    payment      socket          low 2024-01-20 09:57:00+00:00 2024-01-20 10:19:00+00:00             23  4.000000e+00  4.000000e+00  4.066667e+00  4.000000e+00         0.000000
83       user      socket         high 2024-01-20 10:16:00+00:00 2024-01-20 10:21:00+00:00              6  1.900000e+01  1.900000e+01  1.900000e+01  1.720000e+01         0.000000
0       carts         cpu         high 2024-01-20 10:01:00+00:00 2024-01-20 10:02:00+00:00              2  5.410024e+00  3.591809e+00  3.505097e+00  1.368618e+00         0.352111
1       carts         cpu          low 2024-01-20 10:19:00+00:00 2024-01-20 10:20:00+00:00              2  1.355102e+00  1.276872e+00  3.505097e+00  1.368618e+00         0.067036
5       carts  latency-90          low 2024-01-20 09:58:00+00:00 2024-01-20 09:59:00+00:00              2  2.364219e-02  2.343864e-02  1.338379e-01  2.367497e-02         0.009982
4       carts  latency-50         high 2024-01-20 10:20:00+00:00 2024-01-20 10:21:00+00:00              2  1.996569e-02  1.937374e-02  1.929503e-02  1.432997e-02         0.033591
22  catalogue  latency-50         high 2024-01-20 09:58:00+00:00 2024-01-20 09:59:00+00:00              2  3.095219e-03  3.093733e-03  3.093076e-03  3.014332e-03         0.000693
35     orders         cpu          low 2024-01-20 10:18:00+00:00 2024-01-20 10:19:00+00:00              2  1.086319e+00  1.077687e+00  2.829850e+00  1.101655e+00         0.021756
42     orders         mem          low 2024-01-20 10:04:00+00:00 2024-01-20 10:05:00+00:00              2  3.237448e+08  3.235052e+08  3.254620e+08  3.238069e+08         0.000932
43     orders         mem         high 2024-01-20 10:14:00+00:00 2024-01-20 10:15:00+00:00              2  3.255171e+08  3.254690e+08  3.254620e+08  3.238069e+08         0.000169
45     orders      socket          low 2024-01-20 10:08:00+00:00 2024-01-20 10:09:00+00:00              2  1.431667e+01  1.341667e+01  2.210000e+01  1.431667e+01         0.062864
41     orders  latency-90         high 2024-01-20 10:17:00+00:00 2024-01-20 10:18:00+00:00              2  2.234077e+00  2.229248e+00  2.229084e+00  4.723338e-02         0.002235
38     orders  latency-50          low 2024-01-20 10:08:00+00:00 2024-01-20 10:09:00+00:00              2  3.613116e-02  3.517131e-02  1.145421e+00  3.616321e-02         0.027429
39     orders  latency-50         high 2024-01-20 10:17:00+00:00 2024-01-20 10:18:00+00:00              2  1.170386e+00  1.146239e+00  1.145421e+00  3.616321e-02         0.021331
24  catalogue  latency-90         high 2024-01-20 09:58:00+00:00 2024-01-20 09:59:00+00:00              2  4.771395e-03  4.768720e-03  4.767537e-03  4.625798e-03         0.000809
26  catalogue         mem          low 2024-01-20 09:57:00+00:00 2024-01-20 09:58:00+00:00              2  6.046379e+06  6.037504e+06  6.189739e+06  6.048495e+06         0.001817
36     orders      diskio         high 2024-01-20 09:57:00+00:00 2024-01-20 09:58:00+00:00              2  1.841894e+05  3.252287e+04  2.601830e+04  0.000000e+00         0.858742
27  catalogue         mem         high 2024-01-20 10:20:00+00:00 2024-01-20 10:21:00+00:00              2  6.203852e+06  6.190694e+06  6.189739e+06  6.048495e+06         0.002275,      cmdb_id  total_rows_in_window  sum_error_count earliest_error_ts_utc  sum_row_count
0      carts                    46                0                   NaT           1400
1  catalogue                    50                0                   NaT           3874
2     orders                    50                0                   NaT           2908
3    payment                    50                0                   NaT           2414
4       user                    50                0                   NaT          14053)```
```